# 第 11 课｜厚尾与黑天鹅

> **数据身份**：固定随机种子生成的模拟教学数据，不是真实证券行情。  
> **环境**：`pkm-quant` · Python 3.11 · 默认离线运行。  
> **课程原始材料**：`11_youtube-RkHMM_UcsTo/index.html`  
> **所属模块**：P1-M3 · 统计证据与可预测性

## 本节量化问题

正态模型为什么会系统性低估极端收益出现频率？

## 准确定义与策略报告位置

厚尾意味着尾部概率高于同方差正态分布；峰度、分位数和尾部拟合用于描述这种偏离。

- **核心关系**：超额峰度 κ=E[((X-μ)/σ)^4]-3；厚尾表现为远端分位数和极端概率高于正态基准。
- **写入报告**：尾部风险、压力情景和模型风险
- **P2 衔接**：P2 第 48 课和 L4：厚尾诊断与防泄漏验证。


## 运行环境与数据契约

先确认项目、数据目录和数据身份。后续联网数据源必须通过适配层进入，不能改变本 Notebook 的研究计算口径。

In [1]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "data" / "fixtures").exists():
    ROOT = ROOT.parent
DATA_DIR = ROOT / "data" / "fixtures"
OUTPUT_DIR = ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

RANDOM_SEED = 365
print("项目:", ROOT)
print("数据:", DATA_DIR)
print("数据身份: 固定随机种子生成的模拟教学数据，不是真实证券行情")


项目: /Users/mojiawen/Documents/学习资料归档/_暂存/personal_knowledge_manager/量化交易/实验/p1-quant-foundations
数据: /Users/mojiawen/Documents/学习资料归档/_暂存/personal_knowledge_manager/量化交易/实验/p1-quant-foundations/data/fixtures
数据身份: 固定随机种子生成的模拟教学数据，不是真实证券行情


## 比较正态与 t 分布的尾部

同一组收益分别拟合正态和 Student-t。用高分位阈值比较模型给出的双尾概率。

In [2]:
from scipy import stats

daily = pd.read_csv(DATA_DIR / "daily_prices.csv", parse_dates=["date"])
market = daily.loc[daily["symbol"] == "SIM_MARKET"].set_index("date")["adj_close"]
returns = market.pct_change(fill_method=None).dropna()

normal_params = stats.norm.fit(returns)
t_params = stats.t.fit(returns)
threshold = 2.5 * returns.std(ddof=1)
normal_tail = 2.0 * stats.norm.sf(threshold, *normal_params)
t_tail = stats.t.cdf(-threshold, *t_params) + stats.t.sf(threshold, *t_params)
empirical_tail = np.mean(np.abs(returns - returns.mean()) > threshold)

display(pd.Series({
    "empirical_tail": empirical_tail,
    "normal_tail": normal_tail,
    "student_t_tail": t_tail,
    "excess_kurtosis": stats.kurtosis(returns, fisher=True, bias=False),
}))

empirical_tail     0.009272
normal_tail        0.012314
student_t_tail     0.014842
excess_kurtosis    0.367601
dtype: float64

## 方法边界与阶段验收

有限样本的尾部估计误差很大；历史未出现不代表风险不存在。

完成本课后应能：

1. 不看课程材料准确解释核心概念和公式符号；
2. 用小样本手工核对 Notebook 的关键结果；
3. 说明结果能支持什么判断、不能支持什么判断；
4. 指出它在策略报告中的位置，以及仍缺少的数据、成本或稳健性证据。
